In [2]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.0 MB/s eta 0:00:00


In [11]:
import os
import shutil
import random
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive

In [12]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from ultralytics import YOLO

# Dynamically point YOLO toward the Google Drive datastore
data_path = '/content/drive/MyDrive/fruit_ripeness_dataset/archive (1)/dataset'

print("Loading YOLO model...")
model = YOLO('yolo11n-cls.pt')

print("Starting training on native Colab GPU...")
model.train(
    data=data_path,
    epochs=20,
    imgsz=224,
    batch=16,
    device=0  # Leverage allocated Nvidia Cloud GPU
)

Loading YOLO model...
Starting training on native Colab GPU...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/fruit_ripeness_dataset/archive (1)/dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tr

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7de0b15efc80>
curves: []
curves_results: []
fitness: 0.9986627399921417
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9973254799842834, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9986627399921417}
save_dir: PosixPath('/content/runs/classify/train-2')
speed: {'preprocess': 0.1035923380585841, 'inference': 0.5748938178673412, 'loss': 0.00036066756194713217, 'postprocess': 0.000513701525245743}
top1: 0.9973254799842834
top5: 1.0

In [15]:
import numpy as np

# Execute secondary sequential validation to extract exact benchmarking times alongside deep metrics
print("\n--- Detailed Evaluation Metrics (Benchmarking) ---")
metrics = model.val()

print(f"Top-1 Accuracy: {metrics.top1 * 100:.2f}%")
if hasattr(metrics, 'top5'):
    print(f"Top-5 Accuracy: {metrics.top5 * 100:.2f}%")

try:
    cm = metrics.confusion_matrix.matrix
    tp = cm.diagonal()
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

    print("\n--- Deep Metrics (Macro Averaged) ---")
    print(f"Precision: {precision.mean() * 100:.2f}%")
    print(f"Recall:    {recall.mean() * 100:.2f}%")
    print(f"F1-Score:  {f1.mean() * 100:.2f}%")
except Exception as e:
    pass


--- Detailed Evaluation Metrics (Benchmarking) ---
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,537,553 parameters, 0 gradients, 3.2 GFLOPs
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.
train: /content/drive/MyDrive/fruit_ripeness_dataset/archive (1)/dataset/train... found 16219 images in 9 classes ✅ 
val: /content/drive/MyDrive/fruit_ripeness_dataset/archive (1)/dataset/test... found 3739 images in 9 classes ✅ 
test: /content/drive/MyDrive/fruit_ripeness_dataset/archive (1)/dataset/test... found 3739 images in 9 classes ✅ 
val: Fast image access ✅ (ping: 0.7±0.2 ms, read: 75.3±44.2 MB/s, size: 176.4 KB)
val: Scanning /content/drive/MyDrive/fruit_ripeness_dataset/archive (1)/dataset/test... 3739 images, 0 corrupt: 100% ━━━━━━━━━━━━ 3739/3739 784.1Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 234/234 5.7it/s 41.0s
                   all      0.997        

In [16]:
import shutil
import os
from glob import glob

# Safely persist your locally trained weights immediately back to your drive to bypass Colab wipeouts
dest_dir = '/content/drive/MyDrive/models'
os.makedirs(dest_dir, exist_ok=True)

runs = sorted(glob('runs/classify/train*/weights/best.pt'), key=os.path.getmtime)
best_model_path = runs[-1] if runs else 'runs/classify/train/weights/best.pt'
dest_path = os.path.join(dest_dir, 'best_fruit_model.pt')

if os.path.exists(best_model_path):
    shutil.copy2(best_model_path, dest_path)
    print(f"\nSuccessfully persisted best.pt statically onto Google Drive for future inference: {dest_path}")


Successfully persisted best.pt statically onto Google Drive for future inference: /content/drive/MyDrive/models/best_fruit_model.pt
